In [8]:
import warnings; warnings.filterwarnings("ignore")
import time
import jax
jax.config.update("jax_platform_name", "gpu")
from jaxmat.tensors import SymmetricTensor2, dev, main_invariants
from jaxmat.state import AbstractState
from pathlib import Path
import jax.numpy as jnp
import equinox as eqx
import optax
import matplotlib
from jaxmat.utils import default_value
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from jaxmat.nn.icnn import ICNN, QuadICNN, ICNNSkip
import jaxmat.materials as jm
from jaxmat.loader import ImposedLoading, global_solve
from jaxmat.materials.viscoplastic_flows import (
    ArmstrongFrederickHardening, NortonFlow, VoceHardening,
)
from jaxmat.materials.viscoplasticity import ArmstrongFrederickViscoplasticity
from jaxmat.utils import partition_by_node_names

In [9]:
TRUE = dict(
    E=196_000., nu=0.3,
    sig0=150., sigu=300., b=15.,
    C1=50_000., gamma1=300.,
    C2=20_000., gamma2=100.,
    K=10., m=8.,
)

TRUE_PLAST = dict(H_max = 20e3,E=196_000., nu=0.3,sig0=300,H=5e3)

class LinearHardening(eqx.Module):
    sig0: jax.Array = eqx.field(converter=lambda x: jnp.asarray(x, dtype=jnp.float64))  # Limite d'élasticité (MPa)
    H: jax.Array = eqx.field(converter=lambda x: jnp.asarray(x, dtype=jnp.float64))      # Module d'écrouissage (MPa)
    
    def __call__(self, p):
        return self.sig0 + self.H * p



def make_material_plast(p):
    elasticity = jm.LinearElasticIsotropic(E=p["E"], nu=p["nu"])
    hardening = LinearHardening(sig0=p["sig0"], H=p["H"])
    return jm.vonMisesIsotropicHardening(elasticity=elasticity, yield_stress=hardening)

def make_material(p):
    return ArmstrongFrederickViscoplasticity(
        elasticity=jm.LinearElasticIsotropic(E=p["E"], nu=p["nu"]),
        yield_stress=VoceHardening(sig0=p["sig0"], sigu=p["sigu"], b=p["b"]),
        kinematic_hardening=ArmstrongFrederickHardening(
            C    =jnp.array([p["C1"],     p["C2"]]),
            gamma=jnp.array([p["gamma1"], p["gamma2"]]),
        ),
        viscous_flow=NortonFlow(K=p["K"], m=p["m"]),
    )
 
material_true = make_material(TRUE)
material_true = make_material_plast(TRUE_PLAST)

 


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2.  CHARGEMENT : 2 cycles ±1.5 %, 60 pas/demi-cycle
# ═══════════════════════════════════════════════════════════════════════════════
eps_max  = 0.015
n_cycles = 1
Nsteps   = 60
 
half     = jnp.linspace(0, eps_max, Nsteps + 1)
cycle    = jnp.concatenate([half, half[::-1][1:], -half[1:], -half[::-1][1:]])
eps_path = jnp.tile(cycle[1:], n_cycles)
dt_val   = float(eps_max / (1e-3 * Nsteps))
print(f"Chargement : {len(eps_path)} pas, {n_cycles} cycles, dt = {dt_val:.3f} s")

Chargement : 240 pas, 1 cycles, dt = 0.250 s


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4.  DONNÉES SYNTHÉTIQUES (vraie réponse + bruit 2 %)
# ═══════════════════════════════════════════════════════════════════════════════

time  =np.arange(len(eps_path)+1)
print("Génération des données de référence...")
sig_clean = (material_true, eps_path, time)
key       = jax.random.PRNGKey(42)
noise_lvl = 0.02
sig_noisy = sig_clean #+ jax.random.normal(key, sig_clean.shape) * \
            # jnp.max(jnp.abs(sig_clean)) * noise_lvl 
sig_mean = jnp.abs(sig_clean).mean()
sig_noisy_norm =sig_noisy /  sig_mean 
print(f"  σ_xx ∈ [{float(sig_noisy.min()):.1f}, {float(sig_noisy.max()):.1f}] MPa")
print(f"  sig_noisy_norm ∈ [{float(sig_noisy_norm.min()):.1f}, {float(sig_noisy_norm.max()):.1f}]")

eps_mean = jnp.abs(eps_path).mean()
eps_path_norm = eps_path / eps_mean

Génération des données de référence...


AttributeError: 'vonMisesIsotropicHardening' object has no attribute 'constitutive_update_mixed_control'

In [ ]:
InternalState = jm.elastoplasticity.InternalState
